# Atividade Prática 02 — Mecanismo de Atenção, Transformer e Pré-Treinamento GPT

**Disciplina:** Tópicos em Inteligência Artificial (2026.1) — UFPI
**Aluno:** Heitor Moura
**Data:** 2026-04-06

**Objetivo:** Implementar do zero os componentes fundamentais de um LLM: 4 variantes do mecanismo de atenção, bloco Transformer, modelo GPT-like, cálculo de memória, e pré-treinamento com dados em português.

**Referência:** Raschka, S. *Build a Large Language Model (From Scratch)*, 2024 — Capítulos 3-5

---

## 0. Setup

Instalação de dependências e configuração global.

In [ ]:
%pip install -q torch tiktoken transformers plotly pandas tqdm datasets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
from tqdm.auto import tqdm

# Reproducibilidade
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Plotly no Colab/Jupyter
pio.templates.default = "plotly_dark"
try:
    import google.colab
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "notebook"

# ── Helper de formatação ────────────────────────────────────
W = 60  # largura dos separadores

def print_header(title):
    print(f"\n{'═' * W}")
    print(f"  {title}")
    print(f"{'═' * W}")

def print_subheader(title):
    print(f"\n  {title}")
    print(f"  {'─' * (W - 2)}")

def print_kv(pairs, indent=2):
    if not pairs:
        return
    max_k = max(len(k) for k, _ in pairs)
    for k, v in pairs:
        print(f"{' ' * indent}{k:<{max_k}}  {v}")

def print_table(headers, rows, col_widths=None):
    if col_widths is None:
        col_widths = []
        for i, h in enumerate(headers):
            w = len(h)
            for r in rows:
                w = max(w, len(str(r[i])))
            col_widths.append(w + 2)
    # header
    line = "  ".join(f"{h:>{col_widths[i]}}" for i, h in enumerate(headers))
    print(f"  {line}")
    print(f"  {'  '.join('─' * w for w in col_widths)}")
    for r in rows:
        line = "  ".join(f"{str(r[i]):>{col_widths[i]}}" for i in range(len(headers)))
        print(f"  {line}")

print_header("Setup concluído")
print_kv([("Device:", str(device)), ("PyTorch:", torch.__version__)])

---

# 1. Mecanismo de Atenção

Implementação incremental das 4 estratégias de atenção usadas em LLMs, partindo da versão mais simples até a Multi-Head Attention usada no GPT-2.

**Referência:** Raschka, Cap. 3 — *Coding Attention Mechanisms*

### 1.1 Self-Attention sem Pesos Treináveis

A forma mais básica: cada token calcula similaridade com todos os outros via dot product, normaliza com softmax e gera um context vector como média ponderada. Sem nenhum parâmetro treinável.

In [ ]:
class SimpleSelfAttention:
    """Self-attention sem pesos treináveis — apenas dot product + softmax."""

    def __call__(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        scores = x @ x.T                          # (seq_len, seq_len)
        attention_weights = F.softmax(scores, dim=-1)
        context_vectors = attention_weights @ x    # (seq_len, d_model)
        return context_vectors, attention_weights


# ── Input de exemplo (reutilizado em todas as variantes) ────
sentence = "O gato sentou no tapete"
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(sentence)
tokens = [tokenizer.decode([t]) for t in token_ids]

vocab_size = 50257
d_model = 8
seq_len = len(token_ids)

torch.manual_seed(42)
embed = nn.Embedding(vocab_size, d_model)
x = embed(torch.tensor(token_ids))

# ── Execução ────────────────────────────────────────────────
simple_attn = SimpleSelfAttention()
ctx_simple, w_simple = simple_attn(x)

print_header("1.1 Self-Attention sem Pesos Treináveis")
print_kv([
    ("Frase:", f"'{sentence}'"),
    ("Tokens:", str(tokens)),
    ("Input shape:", f"{tuple(x.shape)}  (seq_len={seq_len}, d_model={d_model})"),
    ("Weights shape:", f"{tuple(w_simple.shape)}  (seq_len × seq_len)"),
    ("Output shape:", f"{tuple(ctx_simple.shape)}"),
])
print_subheader("Attention Weights (amostra: token 0)")
for i, t in enumerate(tokens):
    bar = "█" * int(w_simple[0, i].item() * 30)
    print(f"    {t:>8s}  {w_simple[0, i]:.4f}  {bar}")

### 1.2 Self-Attention com Pesos Treináveis (Q, K, V)

Introduz matrizes de projeção aprendíveis: **Query (Wq)**, **Key (Wk)** e **Value (Wv)**. O modelo aprende *o que procurar* (Q), *o que oferecer* (K) e *o que transmitir* (V). Usa scaled dot-product para estabilidade numérica.

In [ ]:
class TrainableSelfAttention(nn.Module):
    """Self-attention com pesos treináveis Q, K, V."""

    def __init__(self, d_model: int):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        d_k = Q.shape[-1]
        scores = (Q @ K.T) / (d_k ** 0.5)         # scaled dot-product
        attention_weights = F.softmax(scores, dim=-1)
        context_vectors = attention_weights @ V
        return context_vectors, attention_weights


torch.manual_seed(42)
trainable_attn = TrainableSelfAttention(d_model)
ctx_train, w_train = trainable_attn(x)

print_header("1.2 Self-Attention com Pesos Treináveis (Q, K, V)")
print_kv([
    ("Input shape:", f"{tuple(x.shape)}"),
    ("Weights shape:", f"{tuple(w_train.shape)}"),
    ("Output shape:", f"{tuple(ctx_train.shape)}"),
    ("Parâmetros:", f"{sum(p.numel() for p in trainable_attn.parameters()):,} (Wq + Wk + Wv)"),
])
print_subheader("Diferença vs Simple: pesos agora são aprendíveis")
diff = (w_train - w_simple).abs().mean().item()
print(f"    Diferença média nos weights: {diff:.4f}")

### 1.3 Causal Attention (Masked + Dropout)

Em modelos autoregressivos como GPT, cada token só pode ver tokens anteriores e a si mesmo. Implementamos com uma **máscara triangular superior** que bloqueia posições futuras, mais **dropout** para regularização.

In [ ]:
class CausalAttention(nn.Module):
    """Causal (masked) self-attention com dropout."""

    def __init__(self, d_model: int, drop_rate: float = 0.2):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        seq_len = x.shape[0]
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        d_k = Q.shape[-1]
        scores = (Q @ K.T) / (d_k ** 0.5)

        # Máscara causal: bloquear posições futuras
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vectors = attention_weights @ V
        return context_vectors, attention_weights


torch.manual_seed(42)
causal_attn = CausalAttention(d_model, drop_rate=0.2)
causal_attn.eval()
with torch.no_grad():
    ctx_causal, w_causal = causal_attn(x)

print_header("1.3 Causal Attention (Masked + Dropout)")
print_kv([
    ("Input shape:", f"{tuple(x.shape)}"),
    ("Weights shape:", f"{tuple(w_causal.shape)}"),
    ("Output shape:", f"{tuple(ctx_causal.shape)}"),
    ("Dropout rate:", "0.2"),
])
print_subheader("Verificação da máscara causal")
for i, t in enumerate(tokens):
    row = "  ".join(f"{w_causal[i, j]:.2f}" for j in range(seq_len))
    print(f"    {t:>8s} │ {row}")
print(f"\n    ℹ Valores acima da diagonal são 0 (futuro bloqueado)")

### 1.4 Multi-Head Attention

Em vez de uma única atenção, dividimos em **múltiplas heads** paralelas. Cada head opera em um subespaço de dimensão `d_k = d_model / n_heads`, aprendendo padrões complementares. Os outputs são concatenados e projetados de volta à dimensão original.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head causal self-attention (suporta batch, com bias como GPT-2)."""

    def __init__(self, d_model: int, n_heads: int, drop_rate: float = 0.2):
        super().__init__()
        assert d_model % n_heads == 0, "d_model deve ser divisível por n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=True)
        self.W_k = nn.Linear(d_model, d_model, bias=True)
        self.W_v = nn.Linear(d_model, d_model, bias=True)
        self.W_o = nn.Linear(d_model, d_model, bias=True)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        has_batch = x.dim() == 3
        if not has_batch:
            x = x.unsqueeze(0)

        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context = (attention_weights @ V).transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, self.d_model)
        output = self.W_o(context)

        if not has_batch:
            output = output.squeeze(0)
            attention_weights = attention_weights.squeeze(0)

        return output, attention_weights


n_heads = 4
torch.manual_seed(42)
mha = MultiHeadAttention(d_model, n_heads, drop_rate=0.2)
mha.eval()
with torch.no_grad():
    ctx_mha, w_mha = mha(x)

print_header("1.4 Multi-Head Attention")
print_kv([
    ("Input shape:", f"{tuple(x.shape)}"),
    ("Output shape:", f"{tuple(ctx_mha.shape)}"),
    ("Weights shape:", f"{tuple(w_mha.shape)}  (n_heads × seq × seq)"),
    ("n_heads:", str(n_heads)),
    ("d_k (por head):", str(d_model // n_heads)),
    ("Parâmetros:", f"{sum(p.numel() for p in mha.parameters()):,} (Wq + Wk + Wv + Wo + biases)"),
])
print_subheader("Atenção média por head (token 0 → demais)")
for h in range(n_heads):
    vals = "  ".join(f"{w_mha[h, 0, j]:.2f}" for j in range(seq_len))
    print(f"    Head {h}: {vals}")

---

# 2. Bloco Transformer

Combina masked multi-head attention, layer normalization, dropout, feed-forward network e funções de ativação GeLU em uma unidade empilhável.

**Estrutura (pre-norm, como GPT-2):**
```
x → LayerNorm → MultiHeadAttention → Dropout → (+x) → LayerNorm → FFN → (+x)
```

**Referência:** Raschka, Cap. 4 — *Implementing a GPT Model from Scratch*

In [ ]:
class FeedForward(nn.Module):
    """FFN com GeLU e expansion factor 4x."""
    def __init__(self, d_model: int, drop_rate: float = 0.2):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(drop_rate),
        )
    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    """Bloco Transformer com pre-norm (GPT-2 style)."""
    def __init__(self, d_model: int, n_heads: int, drop_rate: float = 0.2):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, drop_rate)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, drop_rate)
        self.drop = nn.Dropout(drop_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.attn(self.ln1(x))
        x = x + self.drop(attn_out)          # residual 1
        x = x + self.ffn(self.ln2(x))        # residual 2
        return x


torch.manual_seed(42)
block = TransformerBlock(d_model, n_heads=4, drop_rate=0.2)
block.eval()
with torch.no_grad():
    block_out = block(x)

print_header("2. Bloco Transformer (pre-norm)")
print_kv([
    ("Input shape:", f"{tuple(x.shape)}"),
    ("Output shape:", f"{tuple(block_out.shape)}"),
    ("Ativação:", "GeLU"),
    ("FFN expansion:", f"{d_model} → {4 * d_model} → {d_model}"),
])
print_subheader("Parâmetros por componente")
components = [
    ("LayerNorm 1", block.ln1), ("MultiHeadAttn", block.attn),
    ("LayerNorm 2", block.ln2), ("FeedForward", block.ffn),
]
total = 0
for name, mod in components:
    n = sum(p.numel() for p in mod.parameters())
    total += n
    print(f"    {name:<16s}  {n:>6,} params")
print(f"    {'─' * 30}")
print(f"    {'Total':<16s}  {total:>6,} params")

---

# 3. Modelo GPT-Like

Empilhamos N blocos Transformer e adicionamos embeddings de token e posição, LayerNorm final e camada de projeção (lm_head) para gerar logits sobre o vocabulário.

**Referência:** Raschka, Cap. 4 — *Implementing a GPT Model from Scratch*

In [ ]:
# Configurações da família GPT-2
GPT2_CONFIG = {
    "vocab_size": 50257, "context_length": 1024,
    "emb_dim": 768, "n_heads": 12, "n_layers": 12, "drop_rate": 0.1,
}

GPT2_CONFIGS = {
    "GPT-2 Small (124M)":  {"vocab_size": 50257, "context_length": 1024, "emb_dim": 768,  "n_heads": 12, "n_layers": 12, "drop_rate": 0.1},
    "GPT-2 Medium (355M)": {"vocab_size": 50257, "context_length": 1024, "emb_dim": 1024, "n_heads": 16, "n_layers": 24, "drop_rate": 0.1},
    "GPT-2 Large (774M)":  {"vocab_size": 50257, "context_length": 1024, "emb_dim": 1280, "n_heads": 20, "n_layers": 36, "drop_rate": 0.1},
    "GPT-2 XL (1.5B)":    {"vocab_size": 50257, "context_length": 1024, "emb_dim": 1600, "n_heads": 25, "n_layers": 48, "drop_rate": 0.1},
}


class GPTModel(nn.Module):
    """Modelo GPT-like completo."""

    def __init__(self, config: dict):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])
        self.drop = nn.Dropout(config["drop_rate"])
        self.blocks = nn.Sequential(*[
            TransformerBlock(config["emb_dim"], config["n_heads"], config["drop_rate"])
            for _ in range(config["n_layers"])
        ])
        self.ln_final = nn.LayerNorm(config["emb_dim"])
        self.lm_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape
        tok_emb = self.token_emb(input_ids)
        pos_emb = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
        x = self.drop(tok_emb + pos_emb)
        x = self.blocks(x)
        x = self.ln_final(x)
        return self.lm_head(x)


torch.manual_seed(42)
model = GPTModel(GPT2_CONFIG)
total_params = sum(p.numel() for p in model.parameters())

print_header("3.1 Arquitetura GPT-2 Small")
print_kv([
    ("Camadas:", str(GPT2_CONFIG["n_layers"])),
    ("Heads:", str(GPT2_CONFIG["n_heads"])),
    ("Dimensão:", str(GPT2_CONFIG["emb_dim"])),
    ("Vocabulário:", f"{GPT2_CONFIG['vocab_size']:,}"),
    ("Contexto:", f"{GPT2_CONFIG['context_length']:,} tokens"),
    ("Parâmetros:", f"{total_params:,}"),
])

In [ ]:
from transformers import GPT2LMHeadModel


def load_gpt2_weights(model: GPTModel, model_name: str = "gpt2") -> GPTModel:
    """Carrega pesos do GPT-2 HuggingFace no modelo custom."""
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_sd = hf_model.state_dict()
    d = model.config["emb_dim"]

    with torch.no_grad():
        model.token_emb.weight.copy_(hf_sd["transformer.wte.weight"])
        model.pos_emb.weight.copy_(hf_sd["transformer.wpe.weight"])

        for i in range(model.config["n_layers"]):
            hp = f"transformer.h.{i}"

            model.blocks[i].ln1.weight.copy_(hf_sd[f"{hp}.ln_1.weight"])
            model.blocks[i].ln1.bias.copy_(hf_sd[f"{hp}.ln_1.bias"])

            # Conv1D(in, out) → nn.Linear(out, in): transpor + split QKV
            qkv_w = hf_sd[f"{hp}.attn.c_attn.weight"]
            qkv_b = hf_sd[f"{hp}.attn.c_attn.bias"]
            q_w, k_w, v_w = qkv_w.split(d, dim=1)
            q_b, k_b, v_b = qkv_b.split(d, dim=0)

            model.blocks[i].attn.W_q.weight.copy_(q_w.T)
            model.blocks[i].attn.W_q.bias.copy_(q_b)
            model.blocks[i].attn.W_k.weight.copy_(k_w.T)
            model.blocks[i].attn.W_k.bias.copy_(k_b)
            model.blocks[i].attn.W_v.weight.copy_(v_w.T)
            model.blocks[i].attn.W_v.bias.copy_(v_b)

            model.blocks[i].attn.W_o.weight.copy_(hf_sd[f"{hp}.attn.c_proj.weight"].T)
            model.blocks[i].attn.W_o.bias.copy_(hf_sd[f"{hp}.attn.c_proj.bias"])

            model.blocks[i].ln2.weight.copy_(hf_sd[f"{hp}.ln_2.weight"])
            model.blocks[i].ln2.bias.copy_(hf_sd[f"{hp}.ln_2.bias"])

            model.blocks[i].ffn.layers[0].weight.copy_(hf_sd[f"{hp}.mlp.c_fc.weight"].T)
            model.blocks[i].ffn.layers[0].bias.copy_(hf_sd[f"{hp}.mlp.c_fc.bias"])
            model.blocks[i].ffn.layers[2].weight.copy_(hf_sd[f"{hp}.mlp.c_proj.weight"].T)
            model.blocks[i].ffn.layers[2].bias.copy_(hf_sd[f"{hp}.mlp.c_proj.bias"])

        model.ln_final.weight.copy_(hf_sd["transformer.ln_f.weight"])
        model.ln_final.bias.copy_(hf_sd["transformer.ln_f.bias"])
        model.lm_head.weight.copy_(hf_sd["transformer.wte.weight"])  # weight tying

    return model, hf_model


# Carregar pesos e verificar topologia
torch.manual_seed(42)
model = GPTModel(GPT2_CONFIG)
model, hf_model = load_gpt2_weights(model)

print_header("3.2 Carregamento de Pesos + Verificação de Topologia")
print(f"  Pesos carregados: gpt2 (HuggingFace)\n")

# Verificação de topologia
comparisons = [
    ("Token Embedding",   "token_emb.weight",            "transformer.wte.weight"),
    ("Pos Embedding",     "pos_emb.weight",              "transformer.wpe.weight"),
    ("Block 0 — LN1",    "blocks.0.ln1.weight",         "transformer.h.0.ln_1.weight"),
    ("Block 0 — Wq",     "blocks.0.attn.W_q.weight",    None),
    ("Block 0 — FFN.0",  "blocks.0.ffn.layers.0.weight","transformer.h.0.mlp.c_fc.weight"),
    ("Final LayerNorm",  "ln_final.weight",             "transformer.ln_f.weight"),
    ("LM Head",          "lm_head.weight",              "lm_head.weight"),
]

custom_sd = dict(model.named_parameters())
hf_sd = dict(hf_model.named_parameters())
rows = []
for label, c_name, h_name in comparisons:
    c_s = str(tuple(custom_sd[c_name].shape))
    h_s = str(tuple(hf_sd[h_name].shape)) if h_name and h_name in hf_sd else "—"
    match = "✓" if c_s == h_s else "≈"
    rows.append((label, c_s, h_s, match))

print_table(["Componente", "Custom", "HuggingFace", ""], rows,
            col_widths=[18, 16, 16, 1])

del hf_model  # liberar memória

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8, top_k=40):
    """Gera texto autoregressivamente a partir de um prompt."""
    model.eval()
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = input_ids[:, -model.config["context_length"]:]
            logits = model(context)[:, -1, :] / temperature
            if top_k > 0:
                top_values, _ = torch.topk(logits, top_k)
                logits = torch.where(logits < top_values[:, -1:], float("-inf"), logits)
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
    return tokenizer.decode(input_ids[0].tolist())


model = model.to(device)
prompt = "The future of artificial intelligence"
generated = generate(model, tokenizer, prompt, max_new_tokens=50)

print_header("3.3 Validação — Geração com Pesos GPT-2")
print(f"  Prompt: '{prompt}'\n")
print(f"  Gerado:")
print(f"  {generated}")

---

# 4. Requisitos de Memória

Função para calcular parâmetros e memória dos modelos GPT usando `.numel()`. O GPT-2 usa **weight tying**: os pesos da camada de embeddings são reutilizados na camada de saída (lm_head), então devem ser descontados do total.

**Referência:** Raschka, Cap. 4

In [ ]:
def calculate_memory(config, dtype="float32"):
    """Calcula parâmetros e memória de um modelo GPT."""
    bytes_per = {"float32": 4, "float16": 2, "bfloat16": 2, "int8": 1}[dtype]
    tmp = GPTModel(config)
    total = sum(p.numel() for p in tmp.parameters())
    tying = tmp.token_emb.weight.numel()   # vocab_size × emb_dim
    unique = total - tying
    del tmp
    return {
        "total": total, "tying": tying, "unique": unique,
        "mb_total": (total * bytes_per) / (1024**2),
        "mb_unique": (unique * bytes_per) / (1024**2),
    }


print_header("4. Requisitos de Memória — Família GPT-2")

rows = []
for name, config in GPT2_CONFIGS.items():
    m = calculate_memory(config, "float32")
    short = name.split("(")[0].strip()
    economia = (1 - m["unique"] / m["total"]) * 100
    rows.append((
        short,
        f"{m['total']:>12,}",
        f"{m['unique']:>12,}",
        f"{m['mb_total']:>8.0f} MB",
        f"{m['mb_unique']:>8.0f} MB",
        f"-{economia:.1f}%",
    ))

print_table(
    ["Modelo", "Params Total", "c/ Tying", "FP32 Total", "FP32 Tying", "Economia"],
    rows,
    col_widths=[14, 14, 14, 10, 10, 8],
)

print(f"\n  ℹ Weight tying: token_emb.weight == lm_head.weight")
print(f"  ℹ Economia maior em modelos menores (embedding ∝ vocab, não escala com profundidade)")

---

# 5. Preparação de Dados em Português

Corpus em português da Wikipedia, tokenizado com BPE (tiktoken). Dataset com **janela deslizante** (sliding window) e DataLoader com os parâmetros especificados.

**Parâmetros:** `max_length=256`, `stride=128` (overlap 50%), `batch_size=4`, `shuffle=True`, `dropout=0.2`

**Referência:** Raschka, Cap. 2 e 5

In [ ]:
from datasets import load_dataset

# ── Coleta do corpus ────────────────────────────────────────
print_header("5.1 Coleta do Corpus — Wikipedia PT")
wiki = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)

corpus_texts = []
total_chars = 0
target_chars = 2_000_000

for article in wiki:
    text = article["text"].strip()
    if len(text) > 200:
        corpus_texts.append(text)
        total_chars += len(text)
        if total_chars >= target_chars:
            break

corpus = "\n\n".join(corpus_texts)

# Tokenizar
token_ids = tokenizer.encode(corpus)

print_kv([
    ("Artigos:", f"{len(corpus_texts):,}"),
    ("Caracteres:", f"{len(corpus):,}"),
    ("Tokens:", f"{len(token_ids):,}"),
    ("Amostra:", f"'{corpus[:100]}...'"),
])

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDataset(Dataset):
    """Dataset com sliding window para pré-treinamento de LLM."""

    def __init__(self, token_ids, max_length=256, stride=128):
        self.input_ids = []
        self.target_ids = []
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i : i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1 : i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


dataset = GPTDataset(token_ids, max_length=256, stride=128)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, drop_last=True)

sample_in, sample_tgt = next(iter(dataloader))

print_header("5.2 Dataset + DataLoader")
print_kv([
    ("Amostras:", f"{len(dataset):,}"),
    ("Batches:", f"{len(dataloader):,}"),
    ("max_length:", "256"),
    ("stride:", "128 (overlap 50%)"),
    ("batch_size:", "4"),
    ("shuffle:", "True"),
])
print_subheader("Validação de shapes")
print_kv([
    ("Input batch:", str(tuple(sample_in.shape))),
    ("Target batch:", str(tuple(sample_tgt.shape))),
])
print_subheader("Sanity check (amostra 0, primeiros 5 tokens)")
print(f"    Input[0, :5]:   {sample_in[0, :5].tolist()}")
print(f"    Target[0, :5]:  {sample_tgt[0, :5].tolist()}")
print(f"    Input[0, 1:6]:  {sample_in[0, 1:6].tolist()}  ← deve ser igual ao target")

---

# 6. Pré-Treinamento

Treinamento por 3 épocas com AdamW e avaliação qualitativa com 5 prompts em português (antes vs depois).

**Configuração:** `epochs=3`, `lr=5e-4`, `weight_decay=0.1`, `dropout=0.2`, `CrossEntropyLoss`, gradient clipping (max_norm=1.0)

**Referência:** Raschka, Cap. 5 — *Pretraining on Unlabeled Data*

In [ ]:
# ── Respostas ANTES do treinamento ──────────────────────────
prompts = [
    "O Brasil é um país",
    "A inteligência artificial pode",
    "O presidente do Brasil",
    "A economia brasileira",
    "O futebol no Brasil é",
]

print_header("6.1 Respostas ANTES do Pré-Treinamento")
respostas_antes = []
for p in prompts:
    resp = generate(model, tokenizer, p, max_new_tokens=40, temperature=0.8)
    respostas_antes.append(resp)
    print(f"\n  Prompt: '{p}'")
    print(f"  → {resp}")

In [ ]:
# ── Training loop ───────────────────────────────────────────
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()
loss_history = []
steps_per_epoch = len(dataloader)

print_header("6.2 Treinamento (3 épocas)")
for epoch in range(3):
    epoch_losses = []
    pbar = tqdm(dataloader, desc=f"Época {epoch+1}/3")
    for input_ids, target_ids in pbar:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = loss_fn(logits.view(-1, logits.size(-1)), target_ids.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_losses.append(loss.item())
        loss_history.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg = sum(epoch_losses) / len(epoch_losses)
    print(f"  Época {epoch+1} — Loss média: {avg:.4f}")

print_subheader("Resumo")
print_kv([
    ("Loss inicial:", f"{loss_history[0]:.4f}"),
    ("Loss final:", f"{loss_history[-1]:.4f}"),
    ("Redução:", f"{((loss_history[0] - loss_history[-1]) / loss_history[0] * 100):.1f}%"),
    ("Total steps:", f"{len(loss_history):,}"),
])

In [ ]:
# ── Loss curve (plotly) ─────────────────────────────────────
steps = list(range(1, len(loss_history) + 1))
window = max(1, len(loss_history) // 50)
loss_smooth = pd.Series(loss_history).rolling(window=window, min_periods=1).mean().tolist()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=steps, y=loss_history, mode="lines", name="Loss (raw)",
    line=dict(color="#6a4c93", width=1), opacity=0.3,
))
fig.add_trace(go.Scatter(
    x=steps, y=loss_smooth, mode="lines", name="Loss (smoothed)",
    line=dict(color="#c084fc", width=3),
))

for epoch in range(1, 4):
    step = epoch * steps_per_epoch
    if step <= len(loss_history):
        fig.add_vline(
            x=step, line=dict(color="#e94560", width=1, dash="dash"),
            annotation_text=f"Época {epoch}", annotation_position="top",
            annotation_font=dict(size=10, color="#e94560"),
        )

fig.update_layout(
    title="Curva de Loss — Pré-Treinamento (3 épocas)",
    xaxis_title="Step", yaxis_title="Loss (CrossEntropy)",
    width=900, height=450,
    legend=dict(x=0.75, y=0.95),
    margin=dict(l=50, r=30, t=60, b=40),
)
fig.show()

In [ ]:
# ── Avaliação: Antes vs Depois ──────────────────────────────
print_header("6.3 Avaliação — Antes vs Depois (3 épocas)")

respostas_depois = []
for p in prompts:
    resp = generate(model, tokenizer, p, max_new_tokens=40, temperature=0.8)
    respostas_depois.append(resp)

for i, p in enumerate(prompts):
    antes = respostas_antes[i][len(p):].strip()
    depois = respostas_depois[i][len(p):].strip()
    print(f"\n  Prompt: \"{p}\"")
    print(f"  {'─' * (W - 2)}")
    print(f"  Antes:  {antes[:120]}")
    print(f"  Depois: {depois[:120]}")

---

# 7. Conclusão

## Resumo dos Resultados

| Etapa | Resultado |
|---|---|
| 1. Atenção | 4 variantes implementadas: simple, trainable (Q/K/V), causal (masked + dropout), multi-head |
| 2. Transformer | Bloco completo com pre-norm, GeLU, residuais, dropout |
| 3. GPT-like | Modelo compatível com pesos do GPT-2 HuggingFace — geração coerente |
| 4. Memória | Família GPT-2 calculada com .numel() e weight tying |
| 5. Dados PT-BR | Dataset com sliding window (max_length=256, stride=128, overlap 50%) |
| 6. Pré-treinamento | Loss reduzida em 3 épocas, melhoria qualitativa nas respostas em PT |

## Referências

- Raschka, S. *Build a Large Language Model (From Scratch)*, Manning, 2024
- Vaswani, A. et al. *Attention Is All You Need*, 2017
- Radford, A. et al. *Language Models are Unsupervised Multitask Learners* (GPT-2), 2019